# StageBridge: Semi-Synthetic Benchmark Demo

This notebook demonstrates StageBridge's ability to learn meaningful niche-aware representations using semi-synthetic data with **ground-truth interaction rules**.

## Key Concepts

Following the AMICI methodology (Yuan & Bhargava 2024), we create semi-synthetic data where:
1. **Spatial proximity** between sender and receiver is explicitly controlled
2. **Downstream genes** are activated ONLY when cells are within interaction range
3. **Ground truth is known**: which cells interact, what genes are activated

This allows rigorous evaluation: does StageBridge learn to attend to relevant senders?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

# StageBridge imports
from stagebridge.benchmarks.amici_semisynthetic import (
    AMICISemiSyntheticGenerator,
    SemiSyntheticConfig,
    InteractionRule,
    create_demo_semisynthetic,
)
from stagebridge.models import StageBridge, StageBridgeConfig

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

## 1. Generate Semi-Synthetic Data

We create a benchmark with:
- 3 cell types: Epithelial (receivers), Macrophage, Fibroblast (senders)
- 2 interaction rules at different length scales:
  - **IL1B-IL1R1** (short-range, 50 microns): Macrophage -> Epithelial
  - **CXCL12-CXCR4** (longer-range, 100 microns): Fibroblast -> Epithelial

In [ ]:
# Configure the generator
config = SemiSyntheticConfig(
    n_cells=2000,
    tissue_size=1000.0,  # microns
    spatial_pattern="gradient",  # Creates interacting and non-interacting zones
    n_genes=500,
    n_latent=40,
    ring_radii=[50, 100, 150, 200],
    seed=42,
)

# Create generator and add interaction rules
generator = AMICISemiSyntheticGenerator(config)
generator.add_default_rules()

# Print interaction rules
print("Interaction Rules:")
print("="*60)
for rule in generator.interaction_rules:
    print(f"\n{rule.interaction_name}")
    print(f"  Sender: {rule.sender_type}")
    print(f"  Receiver: {rule.receiver_type}")
    print(f"  Max distance: {rule.max_distance} microns")
    print(f"  Downstream genes: {rule.downstream_genes}")

In [ ]:
# Generate data
neighborhoods, ground_truth = generator.generate()

print(f"\nGenerated {len(neighborhoods)} cells")
print(f"Interacting cells: {ground_truth.is_interacting.sum()} ({100*ground_truth.is_interacting.mean():.1f}%)")
print(f"Receiver-sender pairs: {len(ground_truth.receiver_sender_pairs)}")

## 2. Visualize Spatial Layout

The gradient pattern creates zones where epithelial cells are near macrophages (interacting) vs far from any senders (non-interacting).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Cell types
ax = axes[0]
colors = {'Epithelial': '#1e3a5f', 'Macrophage': '#c44e52', 'Fibroblast': '#64a860'}
for ct in config.cell_types:
    mask = ground_truth.cell_types == ct
    ax.scatter(
        neighborhoods.loc[mask, 'x'],
        neighborhoods.loc[mask, 'y'],
        c=colors[ct], label=ct, s=10, alpha=0.6
    )
ax.set_xlabel('X (microns)')
ax.set_ylabel('Y (microns)')
ax.set_title('Cell Types')
ax.legend()

# Plot 2: Interacting vs non-interacting
ax = axes[1]
epithelial_mask = ground_truth.cell_types == 'Epithelial'
interact_mask = ground_truth.is_interacting & epithelial_mask
non_interact_mask = ~ground_truth.is_interacting & epithelial_mask

ax.scatter(
    neighborhoods.loc[non_interact_mask, 'x'],
    neighborhoods.loc[non_interact_mask, 'y'],
    c='lightgray', label='Non-interacting', s=10, alpha=0.6
)
ax.scatter(
    neighborhoods.loc[interact_mask, 'x'],
    neighborhoods.loc[interact_mask, 'y'],
    c='#c44e52', label='Interacting', s=15, alpha=0.8
)
ax.set_xlabel('X (microns)')
ax.set_ylabel('Y (microns)')
ax.set_title('Interacting Epithelial Cells (Ground Truth)')
ax.legend()

# Plot 3: Distance to nearest sender
ax = axes[2]
dist = ground_truth.nearest_sender_distance.copy()
dist[~epithelial_mask] = np.nan
dist[dist == np.inf] = np.nan

sc = ax.scatter(
    neighborhoods['x'],
    neighborhoods['y'],
    c=dist,
    cmap='RdYlBu_r',
    s=10,
    alpha=0.7
)
plt.colorbar(sc, ax=ax, label='Distance to nearest sender (microns)')
ax.set_xlabel('X (microns)')
ax.set_ylabel('Y (microns)')
ax.set_title('Sender Proximity (Epithelial only)')

plt.tight_layout()
plt.show()

## 3. Ground Truth Analysis

The ground truth tells us exactly which epithelial cells should be affected by interactions.

In [ ]:
# Analyze receiver-sender pairs
pairs = ground_truth.receiver_sender_pairs

if len(pairs) > 0:
    print("Receiver-Sender Pair Statistics:")
    print("="*60)
    for rule_name in pairs['rule_name'].unique():
        rule_pairs = pairs[pairs['rule_name'] == rule_name]
        print(f"\n{rule_name}:")
        print(f"  Total pairs: {len(rule_pairs)}")
        print(f"  Unique receivers: {rule_pairs['receiver_idx'].nunique()}")
        print(f"  Mean distance: {rule_pairs['distance'].mean():.1f} microns")
        print(f"  Distance range: {rule_pairs['distance'].min():.1f} - {rule_pairs['distance'].max():.1f} microns")

In [ ]:
# Expected attention ring distribution
expected_rings = ground_truth.expected_attention_ring[epithelial_mask]
expected_rings = expected_rings[expected_rings >= 0]

fig, ax = plt.subplots(figsize=(8, 5))
ring_labels = [f'Ring {i+1}\n(0-{config.ring_radii[i]}um)' for i in range(len(config.ring_radii))]
counts = [np.sum(expected_rings == i) for i in range(len(config.ring_radii))]

ax.bar(ring_labels, counts, color='#1e3a5f', edgecolor='white', linewidth=1.2)
ax.set_xlabel('Ring')
ax.set_ylabel('Number of interacting receivers')
ax.set_title('Expected Attention Distribution (Ground Truth)\nWhich ring contains the nearest relevant sender?')

for i, c in enumerate(counts):
    ax.text(i, c + 2, str(c), ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 4. StageBridge Model Architecture

StageBridge uses a **receiver-centered tokenized architecture**:
- Token 0: Receiver cell embedding
- Tokens 1-4: Spatial ring context (pooled via learned ISAB+PMA attention)
- Tokens 5-6: HLCA/LuCA reference embeddings
- Tokens 7-8: Pathway and stats features

The self-attention mechanism should learn to attend more strongly to rings containing relevant senders.

In [ ]:
# Initialize model
model_config = StageBridgeConfig(
    input_dim=config.n_latent,
    hidden_dim=128,
    n_heads=4,
    n_layers=3,
    dropout=0.1,
    use_flash_attention=False,
)

model = StageBridge(model_config)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nArchitecture:")
print(model)

## 5. Prepare Data for Model

Convert the semi-synthetic neighborhoods to StageBridge batch format.

In [ ]:
from stagebridge.loaders.dataset import NicheBatch
from stagebridge.contracts import STAGE_TO_IDX

def neighborhoods_to_batch(df: pd.DataFrame, device='cpu', max_batch=64):
    """Convert neighborhoods DataFrame to NicheBatch for model."""
    df = df.head(max_batch)  # Limit for demo
    n = len(df)
    
    # Receiver: concat HLCA + LuCA
    receiver_hlca = np.array(df['receiver_hlca'].tolist())
    receiver_luca = np.array(df['receiver_luca'].tolist())
    receiver = np.concatenate([receiver_hlca, receiver_luca], axis=1)
    
    # Ring cells
    ring_cells = []
    ring_masks = []
    max_cells_per_ring = 32
    
    for ring_idx in range(4):
        col = f'ring_{ring_idx+1}_cells'
        ring_data = df[col].tolist()
        
        # Pad to max_cells_per_ring
        padded = []
        masks = []
        for cells in ring_data:
            cells = np.array(cells) if len(cells) > 0 else np.zeros((0, config.n_latent))
            n_cells = len(cells)
            if n_cells < max_cells_per_ring:
                pad = np.zeros((max_cells_per_ring - n_cells, config.n_latent))
                cells = np.vstack([cells, pad]) if n_cells > 0 else pad
            elif n_cells > max_cells_per_ring:
                cells = cells[:max_cells_per_ring]
                n_cells = max_cells_per_ring
            padded.append(cells)
            mask = np.zeros(max_cells_per_ring, dtype=bool)
            mask[:min(n_cells, max_cells_per_ring)] = True
            masks.append(mask)
        
        ring_cells.append(torch.tensor(np.array(padded), dtype=torch.float32).to(device))
        ring_masks.append(torch.tensor(np.array(masks), dtype=torch.bool).to(device))
    
    # Stage indices
    stage_idx = [STAGE_TO_IDX.get(s, 0) for s in df['stage']]
    
    return NicheBatch(
        receiver=torch.tensor(receiver, dtype=torch.float32).to(device),
        ring_cells=ring_cells,
        ring_masks=ring_masks,
        hlca=torch.tensor(receiver_hlca, dtype=torch.float32).to(device),
        luca=torch.tensor(receiver_luca, dtype=torch.float32).to(device),
        pathway=None,
        stats=None,
        stage_idx=torch.tensor(stage_idx, dtype=torch.long).to(device),
        donor_ids=df['donor_id'].tolist(),
        cell_ids=df['cell_id'].tolist(),
    )

batch = neighborhoods_to_batch(neighborhoods)
print(f"Batch size: {len(batch)}")
print(f"Receiver shape: {batch.receiver.shape}")
print(f"Ring 1 cells shape: {batch.ring_cells[0].shape}")

## 6. Forward Pass and Attention Analysis

Run the model and examine attention weights. The model should learn to attend more strongly to rings containing relevant senders.

In [ ]:
# Forward pass
model.eval()
with torch.no_grad():
    output = model(batch, return_reconstruction=True)

print(f"Output shape (latent): {output.latent.shape}")
print(f"Reconstruction shape: {output.receiver_reconstruction.shape}")
print(f"Reconstruction loss: {output.reconstruction_loss.item():.4f}")

if output.entropy_loss is not None:
    print(f"Entropy loss: {output.entropy_loss.item():.4f}")

## 7. Summary

This semi-synthetic benchmark enables rigorous evaluation of StageBridge:

1. **Ground truth interactions**: We know exactly which cells interact and what genes should be activated
2. **Expected attention patterns**: We know which ring should receive highest attention for each receiver
3. **Evaluation metrics**:
   - Does attention correlate with ground truth sender proximity?
   - Can the model reconstruct receivers better when nearby senders are present?
   - Are downstream genes correctly identified as interaction-dependent?

### Key Findings

| Metric | Value |
|--------|-------|
| Total cells | {n_cells} |
| Interacting receivers | {n_interacting} ({pct_interacting}%) |
| IL1B-IL1R1 pairs | {n_il1b} |
| CXCL12-CXCR4 pairs | {n_cxcl12} |

In [ ]:
# Final summary
n_cells = len(neighborhoods)
n_interacting = ground_truth.is_interacting.sum()
pct_interacting = 100 * n_interacting / n_cells

pairs = ground_truth.receiver_sender_pairs
n_il1b = len(pairs[pairs['rule_name'] == 'IL1B-IL1R1']) if len(pairs) > 0 else 0
n_cxcl12 = len(pairs[pairs['rule_name'] == 'CXCL12-CXCR4']) if len(pairs) > 0 else 0

print("="*60)
print("SEMI-SYNTHETIC BENCHMARK SUMMARY")
print("="*60)
print(f"Total cells: {n_cells}")
print(f"Interacting receivers: {n_interacting} ({pct_interacting:.1f}%)")
print(f"IL1B-IL1R1 pairs: {n_il1b}")
print(f"CXCL12-CXCR4 pairs: {n_cxcl12}")
print("="*60)